Setting GPU configuration

This ensures that the model runs on a specific GPU (GPU 1) if multiple GPUs are available.

In [ ]:
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]="1" #model will be trained on GPU 1


Installs necessary Python packages:
keras: Deep learning framework for building autoencoders.
scipy: Scientific computing library.
PIL (Pillow): Image processing library.
sklearn: Machine learning utilities.
nibabel: For loading and processing MRI scans.

In [ ]:
!pip install keras scipy PIL sklearn nibabel

cv2: OpenCV library for image processing.
keras.layers: Neural network layers for building the autoencoder.
BatchNormalization: Helps stabilize training by normalizing activations.
ModelCheckpoint: Saves the best model during training.
Optimizers: Different optimization algorithms (Adam, RMSprop, etc.).
regularizers: Used to prevent overfitting.


In [ ]:
import os
import cv2
from keras.layers import Input,Dense,Flatten,Dropout,merge,Reshape,Conv2D,MaxPooling2D,UpSampling2D,Conv2DTranspose
from keras.layers.normalization import BatchNormalization
from keras.models import Model,Sequential
from keras.callbacks import ModelCheckpoint
from keras.optimizers import Adadelta, RMSprop,SGD,Adam
from keras import regularizers
from keras import backend as K


nibabel: Reads MRI images stored in NIfTI format (.nii, .nii.gz).
train_test_split: Splits MRI dataset into training and testing sets.
glob: Finds all MRI image files in a directory.
matplotlib.pyplot: For visualizing MRI images.
Note: sklearn.cross_validation is deprecated; should use sklearn.model_selection.train_test_split.

In [ ]:
import numpy as np
import scipy.misc
import numpy.random as rng
from PIL import Image, ImageDraw, ImageFont
from sklearn.utils import shuffle
import nibabel as nib #reading MR images
from sklearn.cross_validation import train_test_split
import math
import glob
from matplotlib import pyplot as plt
%matplotlib inline


In [ ]:
ff = glob.glob('DIR/*')


 Loading and Preprocessing MRI Images

In [ ]:
images = []
for f in range(len(ff)):
    a = nib.load(ff[f])
    a = a.get_data()
    a = a[:,78:129,:]
    for i in range(a.shape[1]):
        images.append((a[:,i,:]))
print (a.shape)

Reshaping and Normalization

In [ ]:
images = np.asarray(images)
images = images.reshape(-1, 173,173,1)

m = np.max(images)
mi = np.min(images)

Padding the Images

In [ ]:
images = (images - mi) / (m - mi)

temp = np.zeros([1530,176,176,1])
temp[:,3:,3:,:] = images

images = temp

Adds padding to ensure a uniform shape (176x176) for all images.

 Splitting Data into Training and Validation Sets

Splits images into training (80%) and validation (20%) sets.
Both train_X and train_ground are the same because an autoencoder learns to reconstruct inputs.

In [ ]:
from sklearn.model_selection import train_test_split
train_X,valid_X,train_ground,valid_ground = train_test_split(images,
                                                             images,
                                                             test_size=0.2,
                                                             random_state=13)


Displaying Sample Images

In [ ]:
# Shapes of training set
print("Dataset (images) shape: {shape}".format(shape=images.shape))


In [ ]:
plt.figure(figsize=[5,5])

# Display the first image in training data
plt.subplot(121)
curr_img = np.reshape(train_X[0], (176,176))
plt.imshow(curr_img, cmap='gray')

# Display the first image in testing data
plt.subplot(122)
curr_img = np.reshape(valid_X[0], (176,176))
plt.imshow(curr_img, cmap='gray')


Defining Model Hyperparameters

In [ ]:
batch_size = 128
epochs = 300
inChannel = 1
x, y = 176, 176
input_img = Input(shape = (x, y, inChannel))

Building the Convolutional Autoencoder

Encoder:
3 convolutional layers extract features.
MaxPooling downsamples the image (reduces spatial dimensions).
Decoder:
Uses UpSampling2D to upscale the image back to original size.
Final layer outputs a grayscale MRI image (sigmoid activation).

In [ ]:
def autoencoder(input_img):
    #encoder
    #input = 28 x 28 x 1 (wide and thin)
    conv1 = Conv2D(32, (3, 3), activation='relu', padding='same')(input_img) #28 x 28 x 32
    conv1 = BatchNormalization()(conv1)
    conv1 = Conv2D(32, (3, 3), activation='relu', padding='same')(conv1)
    conv1 = BatchNormalization()(conv1)
    pool1 = MaxPooling2D(pool_size=(2, 2))(conv1) #14 x 14 x 32
    conv2 = Conv2D(64, (3, 3), activation='relu', padding='same')(pool1) #14 x 14 x 64
    conv2 = BatchNormalization()(conv2)
    conv2 = Conv2D(64, (3, 3), activation='relu', padding='same')(conv2)
    conv2 = BatchNormalization()(conv2)
    pool2 = MaxPooling2D(pool_size=(2, 2))(conv2) #7 x 7 x 64
    conv3 = Conv2D(128, (3, 3), activation='relu', padding='same')(pool2) #7 x 7 x 128 (small and thick)
    conv3 = BatchNormalization()(conv3)
    conv3 = Conv2D(128, (3, 3), activation='relu', padding='same')(conv3)
    conv3 = BatchNormalization()(conv3)


    #decoder
    conv4 = Conv2D(64, (3, 3), activation='relu', padding='same')(conv3) #7 x 7 x 128
    conv4 = BatchNormalization()(conv4)
    conv4 = Conv2D(64, (3, 3), activation='relu', padding='same')(conv4)
    conv4 = BatchNormalization()(conv4)
    up1 = UpSampling2D((2,2))(conv4) # 14 x 14 x 128
    conv5 = Conv2D(32, (3, 3), activation='relu', padding='same')(up1) # 14 x 14 x 64
    conv5 = BatchNormalization()(conv5)
    conv5 = Conv2D(32, (3, 3), activation='relu', padding='same')(conv5)
    conv5 = BatchNormalization()(conv5)
    up2 = UpSampling2D((2,2))(conv5) # 28 x 28 x 64
    decoded = Conv2D(1, (3, 3), activation='sigmoid', padding='same')(up2) # 28 x 28 x 1
    return decoded


Wraps the autoencoder function inside a Model object.
Uses Mean Squared Error (MSE) as the loss function (minimizing reconstruction error).
Uses RMSprop optimizer, which adapts the learning rate for better convergence.

In [ ]:
autoencoder = Model(input_img, autoencoder(input_img))
autoencoder.compile(loss='mean_squared_error', optimizer = RMSprop())


Prints the architecture, number of layers, and total parameters.

In [ ]:
autoencoder.summary()


Trains the autoencoder using the training dataset (train_X).
Uses validation_data=(valid_X, valid_ground) to monitor performance on unseen data.
Runs for 300 epochs with a batch size of 128.
verbose=1 ensures real-time logging of training progress.

In [ ]:
autoencoder_train = autoencoder.fit(train_X, train_ground, batch_size=batch_size,epochs=epochs,verbose=1,validation_data=(valid_X, valid_ground))


Visualizing Training and Validation Loss

Extracts training loss and validation loss from training history.
Plots loss curves to observe model performance over epochs.
If val_loss increases, it may indicate overfitting.

In [ ]:
loss = autoencoder_train.history['loss']
val_loss = autoencoder_train.history['val_loss']
epochs = range(300)
plt.figure()
plt.plot(epochs, loss, 'bo', label='Training loss')
plt.plot(epochs, val_loss, 'b', label='Validation loss')
plt.title('Training and validation loss')
plt.legend()
plt.show()


Making Predictions on Validation Data

In [ ]:
autoencoder = autoencoder.save_weights('autoencoder_mri.h5')
autoencoder = Model(input_img, autoencoder(input_img))
autoencoder.load_weights('autoencoder_mri.h5')
pred = autoencoder.predict(valid_X)


Displaying Original and Reconstructed MRI Images

In [ ]:
plt.figure(figsize=(20, 4))
print("Test Images")
for i in range(5):
    plt.subplot(1, 5, i+1)
    plt.imshow(valid_ground[i, ..., 0], cmap='gray')
plt.show()    
plt.figure(figsize=(20, 4))
print("Reconstruction of Test Images")
for i in range(5):
    plt.subplot(1, 5, i+1)
    plt.imshow(pred[i, ..., 0], cmap='gray')  
plt.show()


Adding Noise to MRI Scans

In [ ]:
[a,b,c,d]= np.shape(valid_X)
mean = 0
sigma = 0.03
gauss = np.random.normal(mean,sigma,(a,b,c,d))
noisy_images = valid_X + gauss


Denoising the Noisy MRI Scans

In [ ]:
pred_noisy = autoencoder.predict(noisy_images)


Evaluating Image Quality Using PSNR

In [ ]:
plt.figure(figsize=(20, 4))
print("Noisy Test Images")
for i in range(5):
    plt.subplot(1, 5, i+1)
    plt.imshow(noisy_images[i, ..., 0], cmap='gray')
plt.show()    
plt.figure(figsize=(20, 4))
print("Reconstruction of Noisy Test Images")
for i in range(5):
    plt.subplot(1, 5, i+1)
    plt.imshow(pred_noisy[i, ..., 0], cmap='gray')  
plt.show()


In [ ]:
valid_pred = autoencoder.predict(valid_X)
mse =  np.mean((valid_X - valid_pred) ** 2)
psnr = 20 * math.log10( 1.0 / math.sqrt(mse))


In [ ]:
print('PSNR of reconstructed validation images: {psnr}dB'.format(psnr=np.round(psnr,2)))


Computes Mean Squared Error (MSE) between original and reconstructed images.
Computes Peak Signal-to-Noise Ratio (PSNR), a metric for image quality.

In [ ]:
noisy_pred = autoencoder.predict(noisy_images)
mse =  np.mean((valid_X - noisy_pred) ** 2)
psnr_noisy = 20 * math.log10( 1.0 / math.sqrt(mse))


Summary of the MRI Autoencoder Pipeline
Loads MRI images and extracts relevant slices.
Preprocesses images (reshaping, normalizing, padding).
Splits data into training and validation sets.
Defines and trains a convolutional autoencoder to reconstruct MRI images.
Visualizes reconstruction quality for both clean and noisy images.
Uses PSNR to measure reconstruction accuracy.
This autoencoder can be used for denoising MRI scans or detecting anomalies in medical imaging.

In [ ]:
print('PSNR of reconstructed validation images: {psnr}dB'.format(psnr=np.round(psnr_noisy,2)))
